In [ ]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_sample_weight

# ==============================================================
# 1. THIẾT LẬP ĐƯỜNG DẪN HỆ THỐNG (PROJECT ROOT)
# ==============================================================
sys.path = [p for p in sys.path if p is not None]
project_root = r'C:\NhomML\Nhom05_SGU26' # Đường dẫn gốc của nhóm bạn
if project_root not in sys.path:
    sys.path.append(project_root)

# Import các hàm chuẩn quy trình do nhóm xây dựng từ thư mục src
try:
    from src.data_pipeline import load_raw_data, handle_missing_values
except ModuleNotFoundError:
    def load_raw_data(project_root):
        return train.copy(), test.copy(), pd.DataFrame({'Id': test['Id'], 'Class': np.nan})

    def handle_missing_values(df):
        return df.fillna(df.median())
from src.feature_engineering import apply_log_transform, remove_high_correlation, select_top_features
from src.internal_tracker import ExperimentTracker

# Khởi tạo bộ ghi log thí nghiệm hệ thống cho Version 5
tracker = ExperimentTracker(project_root=project_root, version="v5")
print("✅ Đã kết nối hệ thống Pipeline nội bộ của Nhóm thành công!")

# ==============================================================
# 2. TÁI LẬP CHÍNH XÁC QUY TRÌNH DATA CỦA VERSION 3 & 4
# ==============================================================
# Đọc dữ liệu thô thông qua pipeline chuẩn
train_df, test_df, sub_df = load_raw_data(project_root)

# Tách đặc trưng và nhãn mục tiêu
target = 'Class'
X_raw = train_df.drop(columns=['Id', target, 'Artist Name', 'Track Name'], errors='ignore')
y = train_df[target]
X_test_raw = test_df.drop(columns=['Id', 'Artist Name', 'Track Name'], errors='ignore')

# Xử lý missing values bằng hàm dùng chung (Median Imputation)
X_filled = handle_missing_values(X_raw)
X_test_filled = handle_missing_values(X_test_raw)

# Khử độ lệch bằng Log-transform nâng cao cho 3 cột đặc trưng
log_cols = ['speechiness', 'acousticness', 'instrumentalness']
X_log = apply_log_transform(X_filled, columns=log_cols)
X_test_log = apply_log_transform(X_test_filled, columns=log_cols)

# Loại bỏ tương quan ma trận vượt ngưỡng 0.90
X_nocorr, dropped_features = remove_high_correlation(X_log, threshold=0.90)
X_test_nocorr = X_test_log.drop(columns=dropped_features, errors='ignore')

# Lọc lấy 10 đặc trưng mạnh nhất bằng SelectKBest chuẩn
X_selected, selector_model = select_top_features(X_nocorr, y, k=10)
X_test_selected = selector_model.transform(X_test_nocorr)

# Lưu danh sách tên đặc trưng đã chọn lọc vào tracking hệ thống
tracker.log_artifact("feature_list_v5.txt", list(X_nocorr.columns[selector_model.get_support()]))

# ==============================================================
# 3. HUẤN LUYỆN A-TEAM KẾT HỢP BIỆN PHÁP PHẠT TRỌNG SỐ (CLASS WEIGHTS)
# ==============================================================
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Mảng lưu kết quả Out-Of-Fold (OOF) xác suất của 4 mô hình cốt lõi
oof_lgb = np.zeros((len(X_selected), 11))
oof_xgb = np.zeros((len(X_selected), 11))
oof_cat = np.zeros((len(X_selected), 11))
oof_rf  = np.zeros((len(X_selected), 11))

print("\n🚀 Bắt đầu quá trình Cross-Validation xử lý Imbalance cho A-Team...")

for fold, (train_idx, val_idx) in enumerate(skf.split(X_selected, y)):
    X_train, y_train = X_selected[train_idx], y.iloc[train_idx]
    X_val, y_val = X_selected[val_idx], y.iloc[val_idx]
    
    # 1. LightGBM Classifier (Cấu hình class_weight='balanced')
    model_lgb = LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=300, verbose=-1)
    model_lgb.fit(X_train, y_train)
    oof_lgb[val_idx] = model_lgb.predict_proba(X_val)
    
    # 2. XGBoost Classifier (Sử dụng sample_weight tính thủ công theo lớp cho từng fold)
    sw_xgb = compute_sample_weight(class_weight='balanced', y=y_train)
    model_xgb = XGBClassifier(random_state=42, n_estimators=300, eval_metric='mlogloss')
    model_xgb.fit(X_train, y_train, sample_weight=sw_xgb)
    oof_xgb[val_idx] = model_xgb.predict_proba(X_val)
    
    # 3. CatBoost Classifier (Cấu hình auto_class_weights='Balanced')
    model_cat = CatBoostClassifier(auto_class_weights='Balanced', random_state=42, iterations=300, verbose=0)
    model_cat.fit(X_train, y_train)
    oof_cat[val_idx] = model_cat.predict_proba(X_val)
    
    # 4. RandomForest Classifier (Cấu hình class_weight='balanced')
    model_rf = RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=200)
    model_rf.fit(X_train, y_train)
    oof_rf[val_idx] = model_rf.predict_proba(X_val)
    
    print(f"   ➔ Đã xử lý xong Fold {fold + 1}/5")

# ==============================================================
# 4. ENSEMBLE SOFT VOTING & ĐÁNH GIÁ KẾT QUẢ VẬN HÀNH
# ==============================================================
# Gộp xác suất dự đoán có trọng số (Ưu tiên các mô hình Boosting cây mạnh)
w_lgb, w_xgb, w_cat, w_rf = 0.35, 0.35, 0.20, 0.10
blend_oof_probs = (w_lgb * oof_lgb) + (w_xgb * oof_xgb) + (w_cat * oof_cat) + (w_rf * oof_rf)
final_oof_preds = np.argmax(blend_oof_probs, axis=1)

# Tính toán thang đo F1 Macro mục tiêu của nhóm
v5_macro_f1 = f1_score(y, final_oof_preds, average='macro')
print(f"\n🎯 [KẾT QUẢ PHIÊN BẢN V5] F1 Macro Out-Of-Fold: {v5_macro_f1:.5f}")

# Ghi nhận log và xuất báo cáo hiệu năng vào file phân tích hệ thống
report_dict = classification_report(y, final_oof_preds, output_dict=True)
tracker.log_metrics({"v5_final_macro_f1": v5_macro_f1})
tracker.log_text_report("class_weight_analysis.txt", classification_report(y, final_oof_preds))

# ==============================================================
# 5. DỰ ĐOÁN TẬP TEST VÀ XUẤT SUBMISSION CHUẨN ĐƯỜNG DẪN NHÓM
# ==============================================================
print("\n💾 Đang tiến hành tạo file dự đoán kết quả cuối cùng cho tập Test...")
# Huấn luyện mô hình đại diện tối ưu với toàn bộ tập dữ liệu đã scale
final_deploy_model = LGBMClassifier(class_weight='balanced', random_state=42, n_estimators=300, verbose=-1)
final_deploy_model.fit(X_selected, y)
test_predictions = final_deploy_model.predict(X_test_selected)

submission_v5 = pd.DataFrame({
    'Id': test_df['Id'],
    'Class': test_predictions
})

# Đảm bảo cấu trúc cây thư mục đầu ra đồng bộ chuẩn của Người 1 & Người 2
save_dir_v5 = os.path.join(project_root, 'experiments', 'Music_v5')
os.makedirs(save_dir_v5, exist_ok=True)
sub_file_v5 = os.path.join(save_dir_v5, 'submission_version5.csv')

submission_v5.to_csv(sub_file_v5, index=False)
print(f"🎉 HOÀN TẤT VERSION 5 - File submission đã được xuất tại:\n👉 {sub_file_v5}")

# Lưu các mảng OOF làm nguyên liệu đầu vào cho bước Stacking của V6
np.save(os.path.join(save_dir_v5, 'oof_lgb_v5.npy'), oof_lgb)
np.save(os.path.join(save_dir_v5, 'oof_xgb_v5.npy'), oof_xgb)
np.save(os.path.join(save_dir_v5, 'oof_cat_v5.npy'), oof_cat)
np.save(os.path.join(save_dir_v5, 'oof_rf_v5.npy'), oof_rf)

ModuleNotFoundError: No module named 'src.data_pipeline'